# SoftMeta Chatterbox TTS Server v0.4.0

Professional multi-audio studio with an age-aware Natural Human Voice Designer and dependency-isolated Parler-TTS environment.

- Select an **L4 GPU** before running the notebook.
- You may use **Run all** safely.
- Engine: `soft-meta/chatterbox-v2@v0.2.1`
- Server and UI: `soft-meta/Chatterbox-TTS-Server@v0.4.0`
- Generate Voice uses an isolated virtual environment because Parler-TTS and Chatterbox require different Transformers versions.


In [ ]:
%%bash
set -euo pipefail

apt-get update -qq
apt-get install -y -qq ffmpeg libsndfile1 git curl ca-certificates lsof

cd /content
rm -rf /content/bin
mkdir -p /content/bin

MICROMAMBA="/content/bin/micromamba"
MICROMAMBA_VERSION="2.6.2-1"
MICROMAMBA_URL="https://github.com/mamba-org/micromamba-releases/releases/download/${MICROMAMBA_VERSION}/micromamba-linux-64"

echo "Downloading micromamba ${MICROMAMBA_VERSION}..."
curl --fail --location --retry 5 --retry-delay 2 --retry-all-errors \
  --connect-timeout 30 "${MICROMAMBA_URL}" --output "${MICROMAMBA}"
chmod +x "${MICROMAMBA}"
"${MICROMAMBA}" --version

if "${MICROMAMBA}" env list | grep -q 'sm311'; then
  "${MICROMAMBA}" env remove -n sm311 -y || true
fi

"${MICROMAMBA}" create -y -n sm311 -c conda-forge python=3.11 pip
echo "Python 3.11 environment is ready."

In [ ]:
%%bash
set -euo pipefail

MM="/content/bin/micromamba"
cd /content
rm -rf chatterbox-v2 Chatterbox-TTS-Server /content/softmeta_voice_env

git clone --branch v0.2.1 --depth 1 https://github.com/soft-meta/chatterbox-v2.git
git clone --branch v0.4.0 --depth 1 https://github.com/soft-meta/Chatterbox-TTS-Server.git

# Main Chatterbox environment.
"$MM" run -n sm311 python -m pip install -U pip wheel
"$MM" run -n sm311 python -m pip install "setuptools==80.9.0"

echo "Installing CUDA PyTorch for Colab L4..."
"$MM" run -n sm311 python -m pip install \
  --index-url https://download.pytorch.org/whl/cu124 \
  torch==2.6.0 torchaudio==2.6.0

echo "Installing the official Chatterbox package..."
"$MM" run -n sm311 python -m pip install --no-cache-dir chatterbox-tts==0.1.7
"$MM" run -n sm311 python -m pip install --force-reinstall "setuptools==80.9.0"

 echo "Installing the SoftMeta engine adapter..."
"$MM" run -n sm311 python -m pip install --no-deps -e /content/chatterbox-v2

echo "Installing the SoftMeta server..."
"$MM" run -n sm311 python -m pip install -r /content/Chatterbox-TTS-Server/requirements-colab.txt
"$MM" run -n sm311 python -m pip install --force-reinstall "setuptools==80.9.0"

# Parler-TTS requires Transformers 4.46.1, while Chatterbox 0.1.7 requires a
# newer incompatible Transformers release. A separate venv shares the main
# CUDA PyTorch installation but keeps Parler's Python dependencies isolated.
echo "Creating the isolated Generate Voice environment..."
"$MM" run -n sm311 python -m venv \
  --system-site-packages \
  /content/softmeta_voice_env

VOICE_PY="/content/softmeta_voice_env/bin/python"
"$VOICE_PY" -m pip install -U pip wheel setuptools
"$VOICE_PY" -m pip install --no-cache-dir \
  -r /content/Chatterbox-TTS-Server/requirements-voice.txt

echo "Installation completed."


In [ ]:
%%bash
set -euo pipefail

/content/bin/micromamba run -n sm311 python - <<'PYVERIFY'
import sys
from importlib.metadata import version

import torch
import torchaudio
import chatterbox
import perth
from softmeta_chatterbox import SoftMetaChatterboxEngine

print("Main environment")
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Torchaudio:", torchaudio.__version__)
print("Transformers:", version("transformers"))
print("Setuptools:", version("setuptools"))
print("PerTh watermarker callable:", callable(getattr(perth, "PerthImplicitWatermarker", None)))
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise SystemExit("CUDA is unavailable. Change the Colab runtime to an L4 GPU.")
if not callable(getattr(perth, "PerthImplicitWatermarker", None)):
    raise SystemExit("PerTh failed to load. Restart the runtime and run from the first cell.")

print("GPU:", torch.cuda.get_device_name(0))
print("Official Chatterbox package:", chatterbox.__file__)
runtime = SoftMetaChatterboxEngine(device="auto")
print("SoftMeta engine device:", runtime.device)
print("Main environment verification passed.")
PYVERIFY

/content/softmeta_voice_env/bin/python - <<'PYVOICE'
from importlib.metadata import version
import torch
import parler_tts
from parler_tts import ParlerTTSForConditionalGeneration

print("\nGenerate Voice environment")
print("Parler-TTS:", version("parler-tts"))
print("Transformers:", version("transformers"))
print("Shared PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Parler import:", ParlerTTSForConditionalGeneration.__name__)

if version("parler-tts") != "0.2.3":
    raise SystemExit("Expected parler-tts 0.2.3 in the isolated environment.")
if version("transformers") != "4.46.1":
    raise SystemExit("Expected Transformers 4.46.1 in the isolated voice environment.")
if not torch.cuda.is_available():
    raise SystemExit("The isolated voice environment cannot access the L4 GPU.")

print("Generate Voice environment verification passed.")
PYVOICE


In [ ]:
import os
import signal
import socket
import subprocess
import time
from pathlib import Path
from IPython.display import HTML, display

PORT = 8004
PROJECT = Path('/content/Chatterbox-TTS-Server')
LOG = Path('/content/softmeta_chatterbox_v040.log')
PID_FILE = Path('/content/softmeta_chatterbox_v040.pid')
MM = '/content/bin/micromamba'

if PID_FILE.exists():
    try:
        os.kill(int(PID_FILE.read_text().strip()), signal.SIGTERM)
        time.sleep(1)
    except Exception:
        pass
subprocess.run(f"lsof -t -i:{PORT} | xargs -r kill -9", shell=True, check=False)
LOG.unlink(missing_ok=True)

env = {
    **os.environ,
    'PYTHONUNBUFFERED': '1',
    'HF_HOME': '/content/hf_home',
    'HF_HUB_CACHE': '/content/hf_home/hub',
    'TRANSFORMERS_CACHE': '/content/hf_home/transformers',
    'SOFTMETA_DEVICE': 'cuda',
    'SOFTMETA_MODEL': 'chatterbox',
    'SOFTMETA_VOICE_PYTHON': '/content/softmeta_voice_env/bin/python',
}
Path(env['HF_HOME']).mkdir(parents=True, exist_ok=True)

log_handle = LOG.open('w', encoding='utf-8', errors='replace')
process = subprocess.Popen(
    [MM, 'run', '-n', 'sm311', 'python', '-u', 'start.py'],
    cwd=PROJECT,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,
)
PID_FILE.write_text(str(process.pid), encoding='utf-8')

def port_open() -> bool:
    try:
        with socket.create_connection(('127.0.0.1', PORT), timeout=.5):
            return True
    except OSError:
        return False

print('Starting SoftMeta Chatterbox TTS Server...')
for _ in range(300):
    if process.poll() is not None:
        log_handle.flush()
        raise RuntimeError(LOG.read_text(errors='replace')[-16000:])
    if port_open():
        break
    time.sleep(1)
else:
    raise TimeoutError('The server did not open port 8004. Run the log cell below.')

# Verify the actual web page before showing the Colab proxy link.
from urllib.request import urlopen
from urllib.error import HTTPError, URLError

try:
    with urlopen(f'http://127.0.0.1:{PORT}/', timeout=10) as response:
        page_status = response.status
        page_preview = response.read(250).decode('utf-8', errors='replace')
except HTTPError as error:
    log_handle.flush()
    error_body = error.read().decode('utf-8', errors='replace')
    recent_log = LOG.read_text(errors='replace')[-16000:] if LOG.exists() else 'No server log.'
    raise RuntimeError(
        f'The server opened port {PORT}, but the home page returned HTTP {error.code}.\n'
        f'Response: {error_body[:2000]}\n\nRecent server log:\n{recent_log}'
    ) from error
except URLError as error:
    log_handle.flush()
    recent_log = LOG.read_text(errors='replace')[-16000:] if LOG.exists() else 'No server log.'
    raise RuntimeError(
        f'The server port opened, but the home page could not be reached: {error}\n\n'
        f'Recent server log:\n{recent_log}'
    ) from error

if page_status != 200 or '<html' not in page_preview.lower():
    raise RuntimeError(
        f'Unexpected home-page response. HTTP {page_status}: {page_preview}'
    )

from google.colab.output import eval_js
url = eval_js(f'google.colab.kernel.proxyPort({PORT})')
html = f'''<p><a href="{url}" target="_blank" style="display:inline-block;padding:13px 19px;background:#5f52e8;color:#fff;border-radius:8px;text-decoration:none;font-weight:700">Open SoftMeta Chatterbox TTS Server</a></p><p style="font-size:13px;color:#667085">The Original model loads in the background. Generate Voice downloads a separate model only when first used.</p>'''
display(HTML(html))
print('Home page check: HTTP 200 OK')
print('Server PID:', process.pid)
print('Server log:', LOG)


## Recent server log

In [ ]:
from pathlib import Path
log = Path('/content/softmeta_chatterbox_v040.log')
print(log.read_text(errors='replace')[-20000:] if log.exists() else 'No server log yet.')

## Optional: Stop the server

Leave `STOP_SERVER` disabled during normal use. Enable it only when you intentionally want to stop the web server.


In [ ]:
STOP_SERVER = False  # @param {type:"boolean"}

import os
import signal
import subprocess
from pathlib import Path

pid_file = Path('/content/softmeta_chatterbox_v040.pid')

if not STOP_SERVER:
    print('Server remains running. Set STOP_SERVER to True only when you want to stop it.')
else:
    if pid_file.exists():
        try:
            os.kill(int(pid_file.read_text().strip()), signal.SIGTERM)
        except Exception:
            pass
        pid_file.unlink(missing_ok=True)
    subprocess.run('lsof -t -i:8004 | xargs -r kill -9', shell=True, check=False)
    print('Server stopped.')
